In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable
from datetime import datetime

start_time = datetime.now()

In [0]:
def audit_log(run_id, schema_name, table_name, start_time, status, row_count, error):
    end_time = datetime.now()
    log = {
        'run_id': run_id,
        'schema_name': schema_name,
        'table_name': table_name,
        'start_time': str(start_time),
        'end_time': str(end_time),
        'duration': (end_time - start_time).total_seconds(),
        'status': status,
        'rows_written': int(row_count or 0),
        'error_message': error
    }
    schema = ("run_id string, schema_name string, table_name string, "
              "start_time string, end_time string, duration double, "
              "status string, rows_written bigint, error_message string")

    spark.createDataFrame([log], schema) \
         .write.mode('append').saveAsTable('retail.ops.audit_logs')

def update_watermark(schema, table, latest_watermark):
    if latest_watermark is None:
        return
    spark.sql(f"""
        UPDATE retail.ops.watermark
        SET last_watermark = TIMESTAMP '{latest_watermark}'
        WHERE schema_name = '{schema}' AND table_name = '{table}'
    """)

In [0]:
dbutils.widgets.text('catalog', 'retail', 'Catalog')
dbutils.widgets.text('schema', 'silver', 'Schema')
dbutils.widgets.text('table', 'orders', 'Table')
dbutils.widgets.text('run_id', 'test', 'Run ID')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table = dbutils.widgets.get('table')
run_id = dbutils.widgets.get('run_id')

In [0]:
latest_watermark= spark.sql(f'''
          SELECT MAX(last_watermark) FROM {catalog}.ops.watermark
          WHERE schema_name = '{schema}' AND table_name = '{table}'
          ''').first()[0]

In [0]:
latest_watermark

In [0]:
src_table_name = f'{catalog}.bronze.{table}'
bronze_batch = (
    spark.read.table(src_table_name) 
    .filter(F.col('_ingested_at') > latest_watermark)
)

### Transformation

In [0]:
## transformation
orders_df = (
    bronze_batch    
    .withColumn('txn_id', F.trim('txn_id'))
    .withColumn('customer_id', F.initcap(F.trim('customer_id')))
    .withColumn('txn_ts',  F.initcap(F.trim('txn_ts')))
    .withColumn('amount', F.round('amount', 2))
    .withColumnRenamed('_ingested_at', '_bronze_ingested_at')
    .withColumnRenamed('run_id','_bronze_run_id')
    .withColumn('_silver_updated_at', F.current_timestamp())   
)

## Dedupe over customer_ids
w = Window.partitionBy('txn_id').orderBy(F.col('_bronze_ingested_at').desc(), F.col('file_modification_time').desc())
orders_df = (
    orders_df
    .withColumn('rn', F.row_number().over(w))
    .filter(F.col('rn')==1)
    .drop('rn', 'file_modification_time')
)

In [0]:
display(orders_df.limit(10))

### Data Quality Checks

In [0]:
dq_check_conditions = {
    'null transaction id': "txn_id IS NULL",
    'null customer id': "customer_id IS NULL",
    'null transaction timestamp': "txn_ts IS NULL",
    'null amount': "amount IS NULL",
    'invalid amount': "amount < 0"
}

valid_df = orders_df

if not orders_df.isEmpty():

    dq_checks = (
        orders_df.withColumn('dq_checks', F.array_compact(F.array(*[F.when(F.expr(cond), F.lit(name)) for name, cond in dq_check_conditions.items()])))
    )

    valid_df = (
        dq_checks.filter(F.size('dq_checks')==0)
        .drop('dq_checks')
    )

    invalid_df = dq_checks.filter(F.size('dq_checks')>0)

    (
        invalid_df
        .withColumn('_quarantined_at', F.current_timestamp())
        .write.mode('append').saveAsTable(f'{catalog}.{schema}.dq_{table}')
    )

### Write the incremental Data

In [0]:
STATUS = None
error = None
rows_written = 0

try:

    if valid_df.isEmpty():
        STATUS = 'SKIPPED'
    else:

        scd_columns = ['txn_id', 'customer_id', 'txn_ts', 'amount', 'currency', 'payment_method', 'status']

        src_data = (
            valid_df
            .withColumn('row_hash', F.sha2(F.concat_ws('||', *scd_columns), 256))
        )

        ## If data is there then check if table exists if not write the new data
        target_table_name = f'{catalog}.{schema}.{table}'

        if not spark.catalog.tableExists(target_table_name):

            (
                src_data
                .write.mode('append')
                .saveAsTable(target_table_name)
            )

            rows_written = src_data.count()

        else:
            ## So here the data already exists
            ## first deactivate the existing records if any (is_current=false)
            ## then append the new records

            target = DeltaTable.forName(spark, target_table_name)

            (
                target.alias('t')
                .merge(
                    src_data.alias('s'),
                    't.txn_id = s.txn_id'
                )
                .whenMatchedUpdateAll(
                    condition = 't.row_hash <> s.row_hash AND s._bronze_ingested_at > t._bronze_ingested_at'
                )
                .whenNotMatchedInsertAll()
                .execute()
            )

            rows_written = src_data.count()

        STATUS = 'COMPLETED'

        silver_watermark = src_data.select(F.max('_bronze_ingested_at')).first()[0]
        if silver_watermark is not None:
            update_watermark(schema, table, silver_watermark)

except Exception as e:
    error = str(e)
    STATUS = 'FAILED'
    raise

finally:
    audit_log(
        run_id=run_id,
        schema_name=schema,
        table_name=table,
        start_time=start_time,
        status=STATUS,
        row_count=rows_written,
        error=error
    )